In [31]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm
from scipy import ndimage
from scipy.optimize import curve_fit
import os
import sys
from scipy.interpolate import interp1d
import corner
from radmc3dPy import image
from radmc3dPy.analyze import *
# from CB68.data_dict import data_dict
from make_conti import generate_model
sys.path.append("..")
from profile_radial.make_disk import generate_disk

In [32]:
sizeau = 80
npix = 500
pixel_area = (sizeau/npix/140)**2
beam_axis = [0.0363, 0.0274]
beam_area = beam_axis[0]*beam_axis[1]*np.pi/(4*np.log(2))
r_axis = np.linspace(-(sizeau//2), sizeau//2, npix, endpoint=True)

In [33]:
edisk_radial = np.load("edisk_radial.npz")
i_r_obs = edisk_radial["i_r"]

In [34]:
def rotate_image(image, posang):
  if isinstance(image, np.ndarray)!=True:
    image.imageJyppix= ndimage.rotate(image.imageJyppix, posang, reshape=False, axes=(1, 0))
    image.imageJyppix = np.nan_to_num(image.imageJyppix, nan=0)
    return image.imageJyppix[:,:,0]
  else:
    image = ndimage.rotate(image, posang, reshape=False, axes=(1, 0))
    image = np.nan_to_num(image, nan=0)
    return image

def radial_intensity(image_array, center, width):
  if center is None:
    peak_idx_x, peak_idx_y = np.unravel_index(np.argmax(image_array, axis=None), image_array.shape)
    center = peak_idx_y
  radial_profile = np.mean(image_array[:, center-width//2:center+width//2], axis=1)
  return radial_profile

def get_radial_profile(image, beam_axis, posang=45, center=None, width=10):
  conv_image = image.imConv(dpc=140, fwhm=beam_axis, pa=-69.4)
  conv_image = rotate_image(conv_image, posang)
  conv_image *= beam_area/pixel_area/(140**2)
  i_r = radial_intensity(conv_image, center, width)
  return i_r

def chi(i_r_model, i_r_obs):
    if len(i_r_model) != len(i_r_obs):
        interp_func = interp1d(np.linspace(0, 1, len(i_r_model)), i_r_model, kind='cubic')
        i_r_model = interp_func(np.linspace(0, 1, len(i_r_obs)))
    return np.sum(((i_r_model - i_r_obs)**2)/(21e-6**2))

def plot_i_r(i_r_model, a, L_star, Q, mdot, heat='radiation'):
    plt.plot(r_axis, i_r_model*1e3, label=" radiation model")
    plt.plot(edisk_radial["r_axis"], edisk_radial["i_r"]*1e3, label="eDisk", linestyle="--")
    plt.xlim((-(sizeau//2), sizeau//2))
    plt.ylim(bottom=0)
    plt.xlabel("Offset (AU)")
    plt.ylabel("Intensity (mJy/beam)")
    plt.title(f"Continuum radial profile along the major axis")
    plt.legend()
    plt.savefig(f"./figures/i_r/{heat}/a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}.pdf", transparent=True)
    plt.close("all")

def plot_image(image, beam_axis, posang, a, L_star, Q, mdot, heat='radiation'):
    conv_image = image.imConv(dpc=140, fwhm=beam_axis, pa=-69.4)
    conv_image = rotate_image(conv_image, posang)
    conv_image *= beam_area/pixel_area/(140**2)
    plt.imshow(conv_image.T, origin='lower', cmap="inferno", vmin=10*21e-6)
    plt.colorbar()
    plt.savefig(f"./figures/image/{heat}/a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}.pdf", transparent=True)
    plt.close("all")


def read_from_dir(dir):
    os.chdir(dir)
    d = readData(dtemp=True, ddens=True, gdens=True, ispec='ch3oh')
    grid = readGrid(wgrid=False)
    os.chdir('..')
    return d, grid

def plot_profile(d, grid):
    nch3oh    = d.ndens_mol[:, :, 0, 0]
    dust      = np.sum(d.rhodust[:, :, 0, :], axis=2)
    t         = np.mean(d.dusttemp[:, :, 0, :], axis=2)
    R, Theta, Phi =  grid.x/au, grid.y, grid.z
    fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(18, 6),
                        subplot_kw={'projection': 'polar'})
    fig.subplots_adjust(left=0.05, right=0.95, top=0.9, bottom=0.1, wspace=0.3, hspace=0.05)
    cmaps = ['BuPu', 'OrRd', 'BuPu']
    titles = [r'$\rho_{dust}$', r'$T$', r'$n_{\mathregular{CH_3OH}}$']
    cbar = [r'log($\rho$) [g$cm^{-3}$]', r'log(T) [K]',r'log($n_{\mathregular{CH_3OH}}$) [$cm^{-3}$]']

    for idx_val, val in enumerate([dust, t, nch3oh]):
        c = ax[idx_val].pcolormesh(Theta-np.pi/2, R, np.log10(val), shading='auto', cmap=cmaps[idx_val])
        ax[idx_val].pcolormesh(Theta+np.pi/2, R, np.log10(val), shading='auto', cmap=cmaps[idx_val])
        if idx_val == 0:
            den = val
            # levels = np.linspace(np.log10(den).min(), np.log10(den).max(), 3)
            levels = [-20, -17, -14]
        ax[idx_val].contour(Theta-np.pi/2, R, np.log10(den), levels=levels, colors='k', linewidths=.7, linestyles='dashed')
        ax[idx_val].contour(Theta+np.pi/2, R, np.log10(den), levels=levels, colors='k', linewidths=.7, linestyles='dashed')
        ax[idx_val].set_xticks([])
        ax[idx_val].set_yticks([])
        fig.colorbar(c, ax=ax[idx_val], orientation='vertical', shrink=0.7).set_label(cbar[idx_val], fontsize=18)
        ax[idx_val].set_title(titles[idx_val], fontsize=26, color='k')


    scale_bar_ax = fig.add_axes([0.48, 0.12, 0.09, 0.02]) # [left, bottom, width, height]
    scale_bar = AnchoredSizeBar(scale_bar_ax.transData,
                                1,  # Size of the scale bar in data coordinates
                                f'{round(R[-1])} AU',  # Label for the scale bar
                                'lower center',  # Location
                                pad=0.1,
                                color='black',
                                frameon=False,
                                size_vertical=0.01,
                                fontproperties=fm.FontProperties(size=12))

    scale_bar_ax.add_artist(scale_bar)
    scale_bar_ax.set_axis_off()

    scale_bar_ax = fig.add_axes([0.15, 0.12, 0.09, 0.02]) # [left, bottom, width, height]
    scale_bar = AnchoredSizeBar(scale_bar_ax.transData,
                                1,  # Size of the scale bar in data coordinates
                                f'{round(R[-1])} AU',  # Label for the scale bar
                                'lower center',  # Location
                                pad=0.1,
                                color='black',
                                frameon=False,
                                size_vertical=0.01,
                                fontproperties=fm.FontProperties(size=12))

    scale_bar_ax.add_artist(scale_bar)
    scale_bar_ax.set_axis_off()

    scale_bar_ax = fig.add_axes([0.8, 0.12, 0.09, 0.02]) # [left, bottom, width, height]
    scale_bar = AnchoredSizeBar(scale_bar_ax.transData,
                                1,  # Size of the scale bar in data coordinates
                                f'{round(R[-1])} AU',  # Label for the scale bar
                                'lower center',  # Location
                                pad=0.1,
                                color='black',
                                frameon=False,
                                size_vertical=0.01,
                                fontproperties=fm.FontProperties(size=12))

    scale_bar_ax.add_artist(scale_bar)
    scale_bar_ax.set_axis_off()

def gaussian(r, I0, r0, sigma):
    return I0 * np.exp(-((r - r0) ** 2) / (2 * sigma ** 2))

def gaussian_fit(i_r, r_axis, extract_index=None):
    I0_guess = np.max(i_r)
    r0_guess = r_axis[np.argmax(i_r)]
    sigma_guess = (np.max(r_axis) - np.min(r_axis)) / 4  # Rough estimate
    p0 = [I0_guess, r0_guess, sigma_guess]
    if extract_index is not None:
        i_r = i_r[extract_index:-extract_index]
        r_axis = r_axis[extract_index:-extract_index]
    popt, pcov = curve_fit(gaussian, r_axis, i_r, p0=p0)
    return popt


# Fix Q=0.5

In [35]:
a_list = [1e-2, 5e-2, 1e-1, 5e-1, 1e0, 1e1]
# a_list = [1e-2, 5e-2]
L_star_list = [1e-1, 5e-1, 1e0, 3e0, 5e0, 1e1]
# Q_list = [0.5, 1, 1.5]
Q_list = [0.5]
mdot_list = [1e-8, 1e-7, 1e-6, 1e-5]
# mdot_list = [1e-7]
# heat_list = ["radiation", "accretion"]
heat_list = ["radiation"]


for idx_a, a in enumerate(a_list):
    for idx_mdot, mdot in enumerate(mdot_list):
        for idx_q, Q in enumerate(Q_list):
            
            for idx_l, L_star in enumerate(L_star_list):
                for heat in heat_list:
                    # try:
                    #     model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}_scat.out')
                    # except:
                    #     model = generate_model(a, L_star, Q, mdot, heat = heat)
                    #     os.system(f"make cleanall")
                    #     model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}_scat.out')
                    model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}_scat.out')
                    i_r_model = get_radial_profile(model_im, beam_axis, posang=45, center=None, width=10)
                    plt.plot(r_axis, i_r_model*1e3, label=r'$L_{*}=$'+f"{L_star}"+r'$L_{\odot}$')
                    
        plt.plot(edisk_radial["r_axis"], edisk_radial["i_r"]*1e3, label="CB68", linestyle="--")
        plt.xlim((-(sizeau//2), sizeau//2))
        plt.ylim(bottom=0)
        plt.xlabel("Offset (AU)")
        plt.ylabel("Intensity (mJy/beam)")
        plt.title(f"Continuum radial profile along the major axis")
        plt.legend()
        plt.savefig(f"./figures/fix_Q_0.5/amax_{a}_mdot_{mdot}.pdf", transparent=True)
        plt.close("all")

Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_0.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.5_Q_0.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_1.0_Q_0.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_3.0_Q_0.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_5.0_Q_0.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_10.0_Q_0.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_0.5_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.5_Q_0.5_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_1.0_Q_0.5_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_3.0_Q_0.5_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_5.0_Q_0.5_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/co

# Fix Q=1.0

In [52]:
a_list = [1e-2, 5e-2, 1e-1, 5e-1, 1e0, 1e1]
# a_list = [1e-2, 5e-2]
L_star_list = [1e-1, 5e-1, 1e0, 3e0, 5e0, 1e1]
# Q_list = [0.5, 1, 1.5]
Q_list = [1]
mdot_list = [1e-8, 1e-7, 1e-6, 1e-5]
# mdot_list = [1e-7]
# heat_list = ["radiation", "accretion"]
heat_list = ["radiation"]


for idx_a, a in enumerate(a_list):
    for idx_mdot, mdot in enumerate(mdot_list):
        for idx_q, Q in enumerate(Q_list):
            
            for idx_l, L_star in enumerate(L_star_list):
                for heat in heat_list:
                    # try:
                    #     model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}_scat.out')
                    # except:
                    #     model = generate_model(a, L_star, Q, mdot, heat = heat)
                    #     os.system(f"make cleanall")
                    #     model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}_scat.out')
                    model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}_scat.out')
                    i_r_model = get_radial_profile(model_im, beam_axis, posang=45, center=None, width=10)
                    plt.plot(r_axis, i_r_model*1e3, label=r'$L_{*}=$'+f"{L_star}"+r'$L_{\odot}$')
                    
        plt.plot(edisk_radial["r_axis"], edisk_radial["i_r"]*1e3, label="CB68", linestyle="--")
        plt.xlim((-(sizeau//2), sizeau//2))
        plt.ylim(bottom=0)
        plt.xlabel("Offset (AU)")
        plt.ylabel("Intensity (mJy/beam)")
        plt.title(f"Continuum radial profile along the major axis")
        plt.legend()
        plt.savefig(f"./figures/fix_Q_1.0/amax_{a}_mdot_{mdot}.pdf", transparent=True)
        plt.close("all")

Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_1_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.5_Q_1_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_1.0_Q_1_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_3.0_Q_1_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_5.0_Q_1_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_10.0_Q_1_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_1_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.5_Q_1_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_1.0_Q_1_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_3.0_Q_1_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_5.0_Q_1_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_10.0_

# Fix Q=1.5 

In [53]:
a_list = [1e-2, 5e-2, 1e-1, 5e-1, 1e0, 1e1]
# a_list = [1e-2, 5e-2]
L_star_list = [1e-1, 5e-1, 1e0, 3e0, 5e0, 1e1]
# Q_list = [0.5, 1, 1.5]
Q_list = [1.5]
mdot_list = [1e-8, 1e-7, 1e-6, 1e-5]
# mdot_list = [1e-7]
# heat_list = ["radiation", "accretion"]
heat_list = ["radiation"]


for idx_a, a in enumerate(a_list):
    for idx_mdot, mdot in enumerate(mdot_list):
        for idx_q, Q in enumerate(Q_list):
            
            for idx_l, L_star in enumerate(L_star_list):
                for heat in heat_list:
                    # try:
                    #     model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}_scat.out')
                    # except:
                    #     model = generate_model(a, L_star, Q, mdot, heat = heat)
                    #     os.system(f"make cleanall")
                    #     model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}_scat.out')
                    model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}_scat.out')
                    i_r_model = get_radial_profile(model_im, beam_axis, posang=45, center=None, width=10)
                    plt.plot(r_axis, i_r_model*1e3, label=r'$L_{*}=$'+f"{L_star}"+r'$L_{\odot}$')
                    
        plt.plot(edisk_radial["r_axis"], edisk_radial["i_r"]*1e3, label="CB68", linestyle="--")
        plt.xlim((-(sizeau//2), sizeau//2))
        plt.ylim(bottom=0)
        plt.xlabel("Offset (AU)")
        plt.ylabel("Intensity (mJy/beam)")
        plt.title(f"Continuum radial profile along the major axis")
        plt.legend()
        plt.savefig(f"./figures/fix_Q_1.5/amax_{a}_mdot_{mdot}.pdf", transparent=True)
        plt.close("all")

Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_1.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.5_Q_1.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_1.0_Q_1.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_3.0_Q_1.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_5.0_Q_1.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_10.0_Q_1.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_1.5_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.5_Q_1.5_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_1.0_Q_1.5_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_3.0_Q_1.5_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_5.0_Q_1.5_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/co

# Fix Mdot

In [56]:
a_list = [1e-2, 5e-2, 1e-1, 5e-1, 1e0, 1e1]
# a_list = [1e-2, 5e-2]
L_star_list = [1e-1, 5e-1, 1e0, 3e0, 5e0, 1e1]
Q_list = [0.5, 1, 1.5]
# Q_list = [1.5]
mdot_list = [1e-8, 1e-7, 1e-6, 1e-5]
# mdot_list = [1e-7]
# heat_list = ["radiation", "accretion"]
heat_list = ["radiation"]


for idx_a, a in enumerate(a_list):

    for idx_q, Q in enumerate(Q_list):
        for idx_mdot, mdot in enumerate(mdot_list):    
            for idx_l, L_star in enumerate(L_star_list):
                for heat in heat_list:
                    # try:
                    #     model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}_scat.out')
                    # except:
                    #     model = generate_model(a, L_star, Q, mdot, heat = heat)
                    #     os.system(f"make cleanall")
                    #     model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}_scat.out')
                    model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}_scat.out')
                    i_r_model = get_radial_profile(model_im, beam_axis, posang=45, center=None, width=10)
                    plt.plot(r_axis, i_r_model*1e3, label=r'$L_{*}=$'+f"{L_star}"+r'$L_{\odot}$')
                    
            plt.plot(edisk_radial["r_axis"], edisk_radial["i_r"]*1e3, label="CB68", linestyle="--")
            plt.xlim((-(sizeau//2), sizeau//2))
            plt.ylim(bottom=0)
            plt.xlabel("Offset (AU)")
            plt.ylabel("Intensity (mJy/beam)")
            plt.title(f"Continuum radial profile along the major axis")
            plt.legend()
            plt.savefig(f"./figures/fix_mdot_{mdot}/amax_{a}_Q_{Q}.pdf", transparent=True)
            plt.close("all")

Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_0.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.5_Q_0.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_1.0_Q_0.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_3.0_Q_0.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_5.0_Q_0.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_10.0_Q_0.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_0.5_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.5_Q_0.5_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_1.0_Q_0.5_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_3.0_Q_0.5_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_5.0_Q_0.5_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/co

# Find how high of Q can saturated intensity (mdot=1e-6)

In [ ]:
a_list = [5e-2, 1e-1]
# a_list = [1e-2, 5e-2]
L_star_list = [5e0]
Q_list = [1.5, 1, 0.5, 0.3, 0.2, 0.1]
# Q_list = [1.5]
mdot_list = [1e-6]
# mdot_list = [1e-7]
# heat_list = ["radiation", "accretion"]
heat_list = ["radiation"]



for idx_mdot, mdot in enumerate(mdot_list):  
    for heat in heat_list:
        for idx_l, L_star in enumerate(L_star_list):
            for idx_a, a in enumerate(a_list):
                for idx_q, Q in enumerate(Q_list):
                    # try:
                    #     model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}_scat.out')
                    # except:
                    #     model = generate_model(a, L_star, Q, mdot, heat = heat)
                    #     os.system(f"make cleanall")
                    #     model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}_scat.out')
                    model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}_scat.out')
                    i_r_model = get_radial_profile(model_im, beam_axis, posang=45, center=None, width=10)
                    plt.plot(r_axis, i_r_model*1e3, label=r'Q='+f"{Q}")
                
                # plt.plot(edisk_radial["r_axis"], edisk_radial["i_r"]*1e3, label="CB68", linestyle="--")
                plt.xlim((-(sizeau//2), sizeau//2))
                plt.ylim(bottom=0)
                plt.xlabel("Offset (AU)")
                plt.ylabel("Intensity (mJy/beam)")
                plt.title(f"Continuum radial profile along the major axis")
                plt.legend()
                plt.savefig(f"./figures/highQ/amax_{a}.pdf", transparent=True)
                plt.close("all")

Reading ./simulation/outfile/conti_a_0.05_Lstar_5.0_Q_1.5_mdot_1e-06_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.05_Lstar_5.0_Q_1_mdot_1e-06_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.05_Lstar_5.0_Q_0.5_mdot_1e-06_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.05_Lstar_5.0_Q_0.3_mdot_1e-06_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.05_Lstar_5.0_Q_0.2_mdot_1e-06_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.05_Lstar_5.0_Q_0.1_mdot_1e-06_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.1_Lstar_5.0_Q_1.5_mdot_1e-06_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.1_Lstar_5.0_Q_1_mdot_1e-06_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.1_Lstar_5.0_Q_0.5_mdot_1e-06_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.1_Lstar_5.0_Q_0.3_mdot_1e-06_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.1_Lstar_5.0_Q_0.2_mdot_1e-06_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.1_

In [59]:
a_list = [1e-2, 5e-2, 1e-1, 5e-1, 1e0, 1e1]
# a_list = [1e-1, 1e0, 1e1]
L_star_list = [1e-1, 5e-1, 1e0, 3e0, 5e0, 1e1]
Q_list = [0.5, 1, 1.5]
# mdot_list = [1e-8, 1e-7, 1e-6, 1e-5]
mdot_list = [1e-8, 1e-7, 1e-6]
heat_list = ["radiation", "accretion"]
# heat_list = ["radiation"]

chi_list_irr = []
idx_list_irr = []
gaussian_list_irr = []

chi_list_acc = []
idx_list_acc = []
gaussian_list_acc = []

for idx_a, a in enumerate(a_list):
    for idx_l, L_star in enumerate(L_star_list):
        for idx_q, Q in enumerate(Q_list):
            for idx_mdot, mdot in enumerate(mdot_list):
                for heat in heat_list:
                    # try:
                    #     model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}_scat.out')
                    # except:
                    #     model = generate_model(a, L_star, Q, mdot, heat = heat)
                    #     os.system(f"make cleanall")
                    #     model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}_scat.out')
                    # model = generate_model(a, L_star, Q, mdot, heat = heat)
                    model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{a}_Lstar_{L_star}_Q_{Q}_mdot_{mdot}_{heat}_scat.out')
                    # plot_image(model_im, beam_axis, posang=45, a=a, L_star=L_star, Q=Q, mdot=mdot, heat=heat)
                    i_r_model = get_radial_profile(model_im, beam_axis, posang=45, center=None, width=10)
                    # plot_i_r(i_r_model, a, L_star, Q, mdot, heat=heat)
                    
                    popt = gaussian_fit(i_r_model, r_axis, extract_index=np.argmin(np.abs(r_axis + 30)))
                    I0_fit, r0_fit, sigma_fit = popt

                    if heat == "radiation":
                        gaussian_list_irr.append((I0_fit, sigma_fit))
                        chi_list_irr.append(chi(i_r_model, edisk_radial["i_r"])*pixel_area/beam_area)
                        idx_list_irr.append((idx_a, idx_l, idx_q, idx_mdot))
                    else:
                        gaussian_list_acc.append((I0_fit, sigma_fit))
                        chi_list_acc.append(chi(i_r_model, edisk_radial["i_r"])*pixel_area/beam_area)
                        idx_list_acc.append((idx_a, idx_l, idx_q, idx_mdot))




Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_0.5_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_0.5_mdot_1e-08_accretion_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_0.5_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_0.5_mdot_1e-07_accretion_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_0.5_mdot_1e-06_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_0.5_mdot_1e-06_accretion_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_1_mdot_1e-08_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_1_mdot_1e-08_accretion_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_1_mdot_1e-07_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_1_mdot_1e-07_accretion_scat.out
Reading ./simulation/outfile/conti_a_0.01_Lstar_0.1_Q_1_mdot_1e-06_radiation_scat.out
Reading ./simulation/outfile/conti_a_0.01_

In [60]:
print(idx_list_irr[chi_list_irr.index(min(chi_list_irr))])
a_best_idx, L_star_best_idx, Q_best_idx, mdot_best_idx = idx_list_irr[chi_list_irr.index(min(chi_list_irr))]
a_best = a_list[a_best_idx]
L_star_best = L_star_list[L_star_best_idx]
Q_best = Q_list[Q_best_idx]
mdot_best = mdot_list[mdot_best_idx]
print(f"Maximum grain size: {a_best} mm")
print(f"Stellar luminosity: {L_star_best} Lsun")
print(f"Toomre Q: {Q_best}")
print(f"Mass accretion rate: {mdot_best} Msun/yr")

(2, 4, 0, 2)
Maximum grain size: 0.1 mm
Stellar luminosity: 5.0 Lsun
Toomre Q: 0.5
Mass accretion rate: 1e-06 Msun/yr


In [61]:
print(idx_list_acc[chi_list_acc.index(min(chi_list_acc))])
a_best_idx, L_star_best_idx, Q_best_idx, mdot_best_idx = idx_list_acc[chi_list_acc.index(min(chi_list_acc))]
a_best = a_list[a_best_idx]
L_star_best = L_star_list[L_star_best_idx]
Q_best = Q_list[Q_best_idx]
mdot_best = mdot_list[mdot_best_idx]
print(f"Maximum grain size: {a_best} mm")
print(f"Stellar luminosity: {L_star_best} Lsun")
print(f"Toomre Q: {Q_best}")
print(f"Mass accretion rate: {mdot_best} Msun/yr")

(4, 2, 0, 2)
Maximum grain size: 1.0 mm
Stellar luminosity: 1.0 Lsun
Toomre Q: 0.5
Mass accretion rate: 1e-06 Msun/yr


In [39]:
# idx_test = (a_list.index(0.1), L_star_list.index(3.0), Q_list.index(0.5), mdot_list.index(1e-6))
# print(idx_list.index(idx_test))
# print(chi_list[idx_list.index(idx_test)])

In [40]:
# radiation_labeled = False
# accretion_labeled = False

# plt.figure(figsize=(10, 6))
# for idx_l, L_star in enumerate(L_star_list):
#     for idx_q, Q in enumerate(Q_list):
#         for idx_a, a in enumerate(a_list):
#             for idx_mdot, mdot in enumerate(mdot_list):
#                 for heat in heat_list:
#                     idx = (a_list.index(a), L_star_list.index(L_star), Q_list.index(Q), mdot_list.index(mdot))
#                     try:
#                         if heat == "radiation":
#                             i0, sigma = gaussian_list_irr[idx_list_irr.index(idx)]
#                             if not radiation_labeled:
#                                 plt.scatter(sigma, i0*1e3, color='r', label="Radiation")
#                                 radiation_labeled = True
#                             else:
#                                 plt.scatter(sigma, i0*1e3, color='r')
#                         else:
#                             i0, sigma = gaussian_list_acc[idx_list_acc.index(idx)]
#                             if not accretion_labeled:
#                                 plt.scatter(sigma, i0*1e3, color='g', label='Accretion')
#                                 accretion_labeled = True
#                             else:
#                                 plt.scatter(sigma, i0*1e3, color='g')
#                     except:
#                         pass
# r_axis_edisk, i_r_edisk = edisk_radial["r_axis"], edisk_radial["i_r"]
# popt_edisk = gaussian_fit(i_r_edisk, r_axis_edisk, extract_index=np.argmin(np.abs(r_axis_edisk + 35)))
# I0_fit_edisk, r0_fit_edisk, sigma_fit_edisk = popt_edisk
# plt.scatter(sigma_fit_edisk, I0_fit_edisk*1e3, color='k', label="CB68")
# plt.xlabel(r"$\sigma$ (AU)", fontsize=14)
# plt.ylabel(r"$I_{0}$ (mJy/beam)", fontsize=14)
# # plt.show()
# plt.legend(fontsize=12)
# plt.title("Comparison of Gaussian fit parameters between different heating mechanisms")
# plt.savefig(f"./figures/gaussian_fit.pdf", transparent=True)
# plt.close("all")

In [41]:
# model_im = image.readImage(fname=f'./simulation/outfile/conti_a_{0.1}_Lstar_{3.0}_Q_{0.5}_mdot_{1e-5}_accretion_scat.out')
# i_r_model = get_radial_profile(model_im, beam_axis, posang=45, center=None, width=10)

In [42]:
# r_index_30au = np.argmin(np.abs(r_axis + 30))
# print(r_index_30au)
# r_index_35au = np.argmin(np.abs(r_axis + 35))
# print(r_index_35au)

In [43]:
# popt = gaussian_fit(i_r_model, r_axis, extract_index=r_index_30au)
# I0_fit, r0_fit, sigma_fit = popt
# print(f"Best fit parameters: I0 = {I0_fit}, r0 = {r0_fit}, sigma = {sigma_fit}")

In [44]:
# plt.figure()
# plt.plot(r_axis, i_r_model, 'bo', label='Data')
# plt.plot(r_axis, gaussian(r_axis, *popt), 'r-', label='Gaussian Fit')
# plt.xlabel('Radius')
# plt.ylabel('Intensity')
# plt.legend()
# plt.title(f"Gaussian Fit: I0={I0_fit:.4f}, sigma={sigma_fit:.2f}")
# plt.show()

In [45]:
# r_axis, i_r_edisk = edisk_radial["r_axis"], edisk_radial["i_r"]
# popt_edisk = gaussian_fit(i_r_edisk, r_axis, extract_index=r_index_35au)
# I0_fit_edisk, r0_fit_edisk, sigma_fit_edisk = popt_edisk
# print(f"Best fit parameters: I0 = {I0_fit_edisk}, r0 = {r0_fit_edisk}, sigma = {sigma_fit_edisk}")

In [46]:
# plt.figure()
# plt.plot(r_axis, i_r_edisk, 'bo', label='Data')
# plt.plot(r_axis, gaussian(r_axis, *popt_edisk), 'r-', label='Gaussian Fit')
# plt.xlabel('Radius')
# plt.ylabel('Intensity')
# plt.legend()
# plt.title(f"Gaussian Fit: I0={I0_fit:.4f}, sigma={sigma_fit:.2f}")
# plt.show()

In [47]:
# disk_best = generate_disk(
#     amax   = 0.05,
#     mstar  = 0.14,
#     mdot   = 1e-8,
#     rd     = 25,
#     Q      = 0.5,
#     l_star = 10,
#     r_star = 1,
#     heat   = "irradiation",
#     dir="./disk_best_irr_mdot_1e-8/"
# )

In [48]:
# d_best, grid = read_from_dir("./disk_best_irr_mdot_1e-8/")
# plot_profile(d_best, grid)

In [49]:
# os.chdir("./disk_best_irr_mdot_1e-8/")
# os.system("radmc3d image npix 500 sizeau 80 incl 70 lambda 1300 posang -45 noline nphot_scat 200000")
# os.chdir("..")

In [50]:
# im = image.readImage(fname=f'./disk_best_irr_mdot_1e-8/image.out')
# im_conv = im.imConv(dpc=140, fwhm=beam_axis, pa=-69.4)

# im_conv = rotate_image(im_conv, 45)

# im_conv *= beam_area/pixel_area/(140**2)
# plt.imshow(im_conv.T, origin='lower', cmap="inferno", vmin=10*21e-6)
# plt.colorbar()


In [51]:
# plt.close("all")